In [ ]:
import os
import pandas as pd

if not os.path.exists('data/ml-latest-small'):
    os.makedirs('data', exist_ok=True)
    # Using shell commands (!) to download and unzip the dataset quietly (-q)
    !curl -O https://files.grouplens.org/datasets/movielens/ml-1m.zip
    !unzip -q ml-1m.zip -d data/
    !rm ml-1m.zip
    print("Dataset downloaded and extracted successfully!")

In [ ]:
!pwd
!ls
!ls data
!ls data/ml-1m
!cat data/ml-1m/movies.dat | head -n 5

In [ ]:
# The 1M dataset uses '::' as a separator and has no header row
users_cols = ['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code']
users_df = pd.read_csv('./data/ml-1m/users.dat', sep='::', engine='python', names=users_cols, encoding='latin-1')

print("User Demographics Sample:")
display(users_df.head())

movies_cols = ['MovieID', 'Title', 'Genres']
movies_df = pd.read_csv('./data/ml-1m/movies.dat', sep='::', engine='python', names=movies_cols, encoding='latin-1')

print("\nMovie Features Sample:")
display(movies_df.head())

ratings_cols = ['UserID', 'MovieID', 'Rating', 'Timestamp']
ratings_df = pd.read_csv('./data/ml-1m/ratings.dat', sep='::', engine='python', names=ratings_cols, encoding='latin-1')

print(f"\nTotal Ratings: {len(ratings_df)}")

In [ ]:
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, MinMaxScaler
import numpy as np

scaler = MinMaxScaler()


merged_df = ratings_df.merge(users_df, on='UserID').merge(movies_df, on='MovieID')

# added new feature for movies
merged_df['Year'] = merged_df['Title'].str.extract(r'\((\d{4})\)').astype(float)
merged_df['Year_Idx'] = LabelEncoder().fit_transform(merged_df['Year'].fillna(0))

# added new feature for movies
movie_stats = ratings_df.groupby('MovieID').agg(
    Movie_Avg_Rating=('Rating', 'mean'),
    Movie_Popularity=('Rating', 'count')
).reset_index()
merged_df = merged_df.merge(movie_stats, on='MovieID')

# added new feature for users
user_stats = ratings_df.groupby('UserID').agg(
    User_Avg_Rating=('Rating', 'mean')
).reset_index()
merged_df = merged_df.merge(user_stats, on='UserID')


merged_df[['Movie_Avg_Rating', 'Movie_Popularity', 'User_Avg_Rating']] = scaler.fit_transform(
    merged_df[['Movie_Avg_Rating', 'Movie_Popularity', 'User_Avg_Rating']]
)

merged_df['Gender_Idx'] = LabelEncoder().fit_transform(merged_df['Gender'])
merged_df['Age_Idx'] = LabelEncoder().fit_transform(merged_df['Age'])

merged_df['Occupation_Idx'] = merged_df['Occupation']

merged_df['Genres_List'] = merged_df['Genres'].str.split('|')
mlb = MultiLabelBinarizer()
genres_multi_hot = mlb.fit_transform(merged_df['Genres_List'])
merged_df['Genres_Encoded'] = list(genres_multi_hot)

In [ ]:
print(merged_df['Year_Idx'].nunique())  

In [ ]:
merged_df.head()

## Content-Based Filtering

In [ ]:

class MovieLensDataset(Dataset):
    def __init__(self, df):
        self.gender = torch.tensor(df['Gender_Idx'].values, dtype=torch.long)
        self.age = torch.tensor(df['Age_Idx'].values, dtype=torch.long)
        self.occupation = torch.tensor(df['Occupation_Idx'].values, dtype=torch.long)
        self.genres = torch.tensor(np.stack(df['Genres_Encoded'].values), dtype=torch.float32)

        self.year = torch.tensor(df['Year_Idx'].values, dtype=torch.long)
        self.movie_avg = torch.tensor(df['Movie_Avg_Rating'].values, dtype=torch.float32)
        self.movie_pop = torch.tensor(df['Movie_Popularity'].values, dtype=torch.float32)
        self.user_avg = torch.tensor(df['User_Avg_Rating'].values, dtype=torch.float32)

        self.ratings = torch.tensor(df['Rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        user_features = {
            'gender': self.gender[idx],
            'age': self.age[idx],
            'occupation': self.occupation[idx],
            'user_avg': self.user_avg[idx],      
        }
        movie_features = {
            'genres': self.genres[idx],            # change from bare tensor to dict
            'year': self.year[idx],
            'movie_avg': self.movie_avg[idx],
            'movie_pop': self.movie_pop[idx],
        }

        return user_features, movie_features, self.ratings[idx]


full_dataset = MovieLensDataset(merged_df)
sample_user, sample_movie, sample_rating = full_dataset[0]

print("Target Rating:", sample_rating.item())
print("User Features:", sample_user)
print("Movie Features:", sample_movie)

In [ ]:
import torch.nn as nn

class UserTower(nn.Module):
    def __init__(self, embedding_dim=16, output_dim=32):
        super().__init__()
        self.age_emb = nn.Embedding(num_embeddings=7, embedding_dim=embedding_dim)
        self.gender_emb = nn.Embedding(num_embeddings=2, embedding_dim=embedding_dim)
        self.occ_emb = nn.Embedding(num_embeddings=21, embedding_dim=embedding_dim)
        
        self.fc1 = nn.Linear(embedding_dim * 3 + 1, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, output_dim)

    def forward(self, user_features):
        age_vec = self.age_emb(user_features['age'])
        gender_vec = self.gender_emb(user_features['gender'])
        occ_vec = self.occ_emb(user_features['occupation'])
        
        x = torch.cat([age_vec, gender_vec, occ_vec, user_features['user_avg'].unsqueeze(1)], dim=1)
        x = self.relu(self.fc1(x))
        return self.fc2(x) 


class MovieTower(nn.Module):
    def __init__(self, num_genres=18, num_years=81, output_dim=32):
        super().__init__()
        self.year_emb = nn.Embedding(num_embeddings=num_years, embedding_dim=8)
        self.fc1 = nn.Linear(num_genres + 8 + 1 + 1, 64)  # 18 genres + 8 year + avg_rating + popularity = 28
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, output_dim)

    def forward(self, movie_features):
        year_vec = self.year_emb(movie_features['year'])
        x = torch.cat([
            movie_features['genres'],
            year_vec,
            movie_features['movie_avg'].unsqueeze(1),
            movie_features['movie_pop'].unsqueeze(1)
        ], dim=1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)


class TwoTowerRecommender(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_tower = UserTower()
        self.movie_tower = MovieTower()

    def forward(self, user_features, movie_features):
        # 1. Pass data through the towers
        v_u = self.user_tower(user_features)
        v_m = self.movie_tower(movie_features)
        
        prediction = torch.sum(v_u * v_m, dim=1)

        prediction = torch.sigmoid(prediction) * 4 + 1  # maps to [1, 5]
        
        return prediction

model = TwoTowerRecommender()
print(model)

In [ ]:
from torch.utils.data import random_split, DataLoader

# Train and Test sets (80% training, 20% testing)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches: {len(train_loader)} | Testing batches: {len(test_loader)}")

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = TwoTowerRecommender().to(device)
criterion = nn.MSELoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_idx, (user_features, movie_features, targets) in enumerate(train_loader):

        movie_features = {k: v.to(device) for k, v in movie_features.items()}
        targets = targets.to(device)

        user_features = {k: v.to(device) for k, v in user_features.items()}
        
        optimizer.zero_grad()
        
        predictions = model(user_features, movie_features)
        
        loss = criterion(predictions, targets)
        
        loss.backward()
        
        optimizer.step()
        
        running_loss += loss.item()
        
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for user_features, movie_features, targets in test_loader:
            movie_features = {k: v.to(device) for k, v in movie_features.items()}
            targets = targets.to(device)
            user_features = {k: v.to(device) for k, v in user_features.items()}
            
            predictions = model(user_features, movie_features)
            loss = criterion(predictions, targets)
            val_loss += loss.item()
            
    epoch_train_loss = running_loss / len(train_loader)
    epoch_val_loss = val_loss / len(test_loader)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train MSE: {epoch_train_loss:.4f} | Val MSE: {epoch_val_loss:.4f}")

In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for user_features, movie_features, targets in test_loader:
        user_features = {k: v.to(device) for k, v in user_features.items()}
        movie_features = {k: v.to(device) for k, v in movie_features.items()}
        
        predictions = model(user_features, movie_features)
        all_preds.extend(predictions.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Core metrics
rmse = np.sqrt(np.mean((all_preds - all_targets) ** 2))
mae = np.mean(np.abs(all_preds - all_targets))

# Accuracy proxy — within 1 star
within_1 = np.mean(np.abs(all_preds - all_targets) <= 1.0)

print(f"RMSE:            {rmse:.4f}")
print(f"MAE:             {mae:.4f}")
print(f"Within-1 Accuracy: {within_1 * 100:.2f}%")

## Collaborative Filtering